# Homework 2 — Vector Search

This notebook is my solution to the **Module 2 homework of [LLM Zoomcamp](https://github.com/DataTalksClub/llm-zoomcamp/tree/main/cohorts/2026/02-vector-search) by DataTalksClub**. It's shared as a walkthrough of how each problem was approached, in case it's useful to others going through the same material.

## What's covered
The focus is on turning text into vectors, searching by semantic similarity, and combining vector search with keyword search — skipping RAG and concentrating purely on the search layer.

## Stack used

| Component | Tool |
|---|---|
| Embeddings | ONNX Runtime (`Xenova/all-MiniLM-L6-v2`) |
| Vector search | `minsearch.VectorSearch` |
| Keyword search | `minsearch.Index` |
| Document source | `gitsource` (GitHub repo reader) |
| Numerical ops | `numpy` |

> API keys are not required for this homework — all embedding runs locally via ONNX.

## Environment Setup

The environment is set up in the homework directory using `uv`. The ONNX Runtime is used instead of `sentence-transformers` — it produces identical vectors but is ~30× lighter, with no PyTorch or CUDA dependency.

Two helper scripts are downloaded from the course repository:
- `download.py` — fetches the ONNX model (`Xenova/all-MiniLM-L6-v2`) from HuggingFace
- `embedder.py` — provides the `Embedder` class with an `encode` / `encode_batch` interface

Run in terminal to set up the project:

```bash
uv init --no-workspace --python 3.13
uv add onnxruntime tokenizers numpy tqdm minsearch gitsource
uv add --dev huggingface-hub jupyter
```

In [1]:
# # Download the two helper scripts from the course repo
# PREFIX="https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed"

# !wget $PREFIX/download.py   # fetches the ONNX model from HuggingFace
# !wget $PREFIX/embedder.py   # Embedder class with encode / encode_batch interface

In [2]:
# # Download the ONNX model weights (Xenova/all-MiniLM-L6-v2) — runs once, cached locally
# !uv run python download.py

## Q1 — Embedding a Query

The `Embedder` wraps the ONNX model and exposes a simple `encode(text)` interface that returns a normalized vector of 384 dimensions.

The query below is embedded and the first element of the resulting vector is inspected.

In [3]:
from embedder import Embedder

embed = Embedder()

query = "How does approximate nearest neighbor search work?"

# Encode the query — returns a normalized float32 vector of length 384
v1 = embed.encode(query)
print(f"Vector dimensions: {len(v1)}")

Vector dimensions: 384


In [4]:
# Q1: What is the first value of the query embedding vector?
print(f"v[0] = {v1[0]:.4f}")

v[0] = -0.0206


## Prepare the Knowledge Base

The course lesson pages are pulled from GitHub using `gitsource`, pinned to commit `8c1834d` so results are reproducible across runs. The `filename_filter` keeps only files inside a module's `lessons/` subfolder, skipping top-level READMEs and other markdown.

There are 72 lesson pages in total — same dataset as Homework 1.

In [5]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",        # pinned commit — ensures reproducibility
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
print(f"Total lesson pages loaded: {len(documents)}")

Total lesson pages loaded: 72


In [6]:
# Preview the first document to confirm the schema: filename + content
documents[:1]

[{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a sim

## Q2 — Cosine Similarity

Since the embedder returns **normalized** vectors, the dot product between any two vectors is equivalent to their cosine similarity — no extra normalization step needed.

The content of `02-vector-search/lessons/07-sqlitesearch-vector.md` is embedded and its cosine similarity with the Q1 query vector is computed.

In [7]:
# Retrieve the content of the target lesson page
target_file = "02-vector-search/lessons/07-sqlitesearch-vector.md"
content = next(doc['content'] for doc in documents if doc.get('filename') == target_file)
print(f"Content length: {len(content)} characters")

Content length: 7219 characters


In [8]:
# Embed the full page content into a 384-dimensional normalized vector
v2 = embed.encode(content)

In [9]:
# Q2: Cosine similarity between the query vector (v1) and the page vector (v2)
# dot product of two normalized vectors = cosine similarity
similarity = v2.dot(v1)
print(f"Cosine similarity: {similarity:.4f}")

Cosine similarity: 0.3611


## Q3 — Chunking and Search by Hand

A full lesson page can cover several topics, which dilutes its embedding — a single vector has to represent everything on the page. Chunking splits each page into smaller overlapping windows so each embedding captures a more focused slice of content.

Each chunk uses a 2 000-character window advancing in 1 000-character steps (1 000-character overlap between consecutive chunks). Every chunk keeps the original `filename` and `start` offset.

The chunks are embedded in batches, stacked into a matrix `X`, and scored against the Q1 query with a simple dot product.

In [10]:
from gitsource import chunk_documents

# Split each lesson page into overlapping 2000-char windows (step=1000 → 1000-char overlap)
chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Total chunks: {len(chunks)}")

Total chunks: 295


In [11]:
# Prepend filename to content so the embedding captures file context as well
texts = [doc["filename"] + " " + doc["content"] for doc in chunks]
print(f"Sample text (first 200 chars):-\n{texts[0][:200]}")

Sample text (first 200 chars):-
01-agentic-rag/lessons/01-intro.md # Introduction

Video: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In this module, we'll build a workin


In [12]:
from tqdm.auto import tqdm
import numpy as np

# Encode all chunks in batches of 50 to avoid memory spikes
batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

# Stack into a 2D matrix: shape = (num_chunks, 384)
X = np.array(X)
print(f"Embedding matrix shape: {X.shape}")

  0%|          | 0/6 [00:00<?, ?it/s]

Embedding matrix shape: (295, 384)


In [13]:
# Q3: Score all chunks against the Q1 query vector and find the top match
scores = X.dot(v1)          # dot product = cosine similarity (vectors are normalized)
idx = np.argmax(scores)     # index of the highest-scoring chunk

print(f"Top chunk filename : {chunks[idx].get('filename')}")
print(f"Score              : {scores[idx]:.4f}")

Top chunk filename : 02-vector-search/lessons/07-sqlitesearch-vector.md
Score              : 0.5686


In [14]:
# Show the top 5 scoring chunks for a broader view of the results
top_indices = np.argsort(scores)[-5:][::-1]

for idx in top_indices:
    chunk = chunks[idx]
    score = scores[idx]
    print(f"Score: {score:.4f} | File: {chunk.get('filename')}")
    print(f"Preview: {chunk.get('content')[:100]}...")
    print("-" * 60)

Score: 0.5686 | File: 02-vector-search/lessons/07-sqlitesearch-vector.md
Preview: rch. We score
the query against every document and pick the top ones. It always finds
the true top m...
------------------------------------------------------------
Score: 0.5047 | File: 01-agentic-rag/lessons/05-search.md
Preview: # Search

Video: [Watch this lesson](https://www.youtube.com/watch?v=GYgpNKiuCJU&list=PL3MmuxUbc_hLZ...
------------------------------------------------------------
Score: 0.4781 | File: 02-vector-search/lessons/01-intro.md
Preview: dding model produces these vectors. It's a neural network
trained to capture meaning, so texts that ...
------------------------------------------------------------
Score: 0.4107 | File: 04-evaluation/lessons/05-search-metrics.md
Preview: k mean the same thing.

Let's calculate it:

```python
cnt = 0

for line in example:
    if 1 in lin...
------------------------------------------------------------
Score: 0.4088 | File: 02-vector-search/lessons/05

## Q4 — Vector Search with minsearch

Manual dot-product scoring works for learning, but in practice a vector search library handles indexing and retrieval efficiently.

`minsearch.VectorSearch` is fitted with the pre-computed embedding matrix `X` and the corresponding chunk metadata. A new query is embedded and searched against the index.

In [15]:
from minsearch import VectorSearch

# Build the vector index from the pre-computed chunk embeddings
vindex = VectorSearch()
vindex.fit(X, chunks)

# Q4: Search with a new query — what is the top-ranked filename?
query = "What metric do we use to evaluate a search engine?"
query_vector = embed.encode(query)

result = vindex.search(query_vector, num_results=1)
print(f"Top result: {result[0].get('filename')}")

Top result: 04-evaluation/lessons/05-search-metrics.md


## Q5 — Text Search vs Vector Search

Vector search retrieves by meaning — it finds semantically related content even when the exact words differ. Keyword search retrieves by exact term matching — it excels with specific names, codes, or rare terminology but misses paraphrases.

Both methods are run against the same query and their top-5 results are compared to identify documents that semantic search surfaces but keyword search misses.

In [16]:
from minsearch import Index

# Build a keyword search index over the same chunks
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(chunks)

In [17]:
query = "How do I store vectors in PostgreSQL?"

# Keyword search — matches on exact terms in the content field
results = index.search(query, num_results=5)
text_res = [res['filename'] for res in results]
print("Text search results:")
for f in text_res:
    print(f"  {f}")

Text search results:
  02-vector-search/lessons/02-embeddings.md
  03-orchestration/lessons/05-rag.md
  02-vector-search/lessons/01-intro.md
  03-orchestration/lessons/05-rag.md
  02-vector-search/lessons/01-intro.md


In [18]:
# Vector search — matches on semantic similarity
query_vector = embed.encode(query)
result = vindex.search(query_vector, num_results=5)
vector_res = [res['filename'] for res in result]
print("Vector search results:")
for f in vector_res:
    print(f"  {f}")

Vector search results:
  02-vector-search/lessons/08-pgvector.md
  02-vector-search/lessons/08-pgvector.md
  02-vector-search/lessons/08-pgvector.md
  02-vector-search/lessons/08-pgvector.md
  02-vector-search/lessons/08-pgvector.md


In [19]:
# Q5: Files returned by vector search but not by keyword search
# These are semantically relevant pages that don't contain the exact query terms
only_in_vector = set(vector_res) - set(text_res)
print(f"In vector but not text: {only_in_vector}")

In vector but not text: {'02-vector-search/lessons/08-pgvector.md'}


## Q6 — Hybrid Search with Reciprocal Rank Fusion (RRF)

Neither vector nor keyword search is universally better — their strengths are complementary. Hybrid search combines both by merging their ranked result lists using **Reciprocal Rank Fusion (RRF)**.

RRF ignores raw scores (which live on different scales) and instead uses only the *rank position* of each document in each list:

```
RRF(d) = Σ  1 / (k + rank(d))
```

- `k = 60` is the standard default from the original RRF paper — it flattens the gap between positions so that a document ranked 1st isn't overwhelmingly favored over one ranked 5th.
- A document appearing in both result lists accumulates a score contribution from each, which is why hybrid search can surface documents that neither method alone would rank first.

In [20]:
def rrf(result_lists, k=60, num_results=5):
    """
    Reciprocal Rank Fusion — merges multiple ranked result lists into one.

    Each document is scored by its rank position across all lists:
        score += 1 / (k + rank)
    Documents appearing in multiple lists accumulate scores from each.
    """
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [21]:
query = "How do I give the model access to tools?"

# Run keyword search
text_results = index.search(query)
print("Text search results:")
for res in text_results:
    print(res.get('filename'))
# print("Text search top result:", text_results[0].get('filename'))

Text search results:
01-agentic-rag/lessons/14-agentic-loop.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/13-function-calling.md
04-evaluation/lessons/02-ground-truth.md
01-agentic-rag/lessons/14-agentic-loop.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/11-agents-intro.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/13-function-calling.md


In [22]:
# Run vector search
query_vector = embed.encode(query)
vector_results = vindex.search(query_vector)
print("Vector search results:")
for res in vector_results:
    print(res.get('filename'))
# print("Vector search top result:", vector_results[0].get('filename'))

Vector search results:
01-agentic-rag/lessons/15-frameworks.md
04-evaluation/lessons/02-ground-truth.md
01-agentic-rag/lessons/16-other-frameworks.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/08-rag-helper.md
01-agentic-rag/lessons/13-function-calling.md
05-monitoring/lessons/02-assistant-setup.md
01-agentic-rag/lessons/07-llm.md
05-monitoring/lessons/02-assistant-setup.md


In [23]:
# Q6: Fuse both result lists with RRF and check the top-ranked document
# The winner may not be first in either individual list —
# it wins by ranking consistently high across both.
results = rrf([vector_results, text_results])
print(f"RRF top result: {results[0].get('filename')}")

RRF top result: 01-agentic-rag/lessons/13-function-calling.md


In [24]:
print(f"RRF top 5 results:")
for res in results:
    print(res.get('filename'))

RRF top 5 results:
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/15-frameworks.md
01-agentic-rag/lessons/14-agentic-loop.md
